## NumPy: продвинутый

In [ ]:
import numpy as np

###Симуляция данных в numpy

Иногда нам нужно сгенерировать массив, заполненный симулированными данными, например, чтобы проверить, как работает какая-то написанная нами функция.


####Рандомизация

Для этих целей в NumPy есть свой подмодуль random, у которого есть следующие вещи:

In [ ]:
arr1 = np.random.rand(2, 2)
print(arr1)

[[0.86868961 0.51243902]
 [0.00097902 0.67181161]]


In [ ]:
arr2 = np.random.randn(2, 2)
print(arr2)

[[-0.64655946  1.17751863]
 [ 1.66718285 -0.15224848]]


In [ ]:
arr3 = np.random.randint(10, 50, size=(2, 2))
print(arr3)

[[22 48]
 [34 22]]


Первая функция генерирует массив заданного размера с числами в полуинтервале [0, 1), распределенными равномерно. Это значит, что все числа выпадают с равной вероятностью.

Вторая функция создает массив, состоящий из рандомных чисел, причем числа могут быть отрицательными и не обязательно от 0 до 1. Распределение здесь Гауссово (стандартное нормальное распределение), то есть середина выпадает существенно чаще, чем края. Среднее значение 0, стандартное отклонение 1.

Третья функция выдает массив целых чисел в заданном диапазоне (первый аргумент означает начало диапазона, включительно, второй - конец, не включительно). Если мы подадим только один позиционный аргумент, началом диападона будет считаться число 0.

Есть также функция для создания случайной выборки из списка или массива:

In [ ]:
fruits = ['apple', 'banana', 'orange']
# Выбираем 5 элементов с заданной вероятностью
selection = np.random.choice(fruits, size=5, p=[0.6, 0.3, 0.1])
print(selection)

['apple' 'banana' 'banana' 'banana' 'apple']


Обратите внимание: если мы укажем вероятности, не дающие в сумме 1 (полную вероятность, соответствующую всему пространству возможных исходов), нам выпадет ошибка!

####Повторяемость и seed

Еще одна функция np.random называется seed. После ее вызова компьютер генерирует псевдослучайные числа на основе стартового числа (зерна). Если зафиксировать seed, то при каждом перезапуске кода вы будете получать абсолютно одинаковые случайные числа. Это критически важно для повторяемости научных экспериментов и тестирования.

In [ ]:
np.random.seed(42)
print(np.random.rand(2))  # При каждом запуске этот результат будет идентичным

[0.37 0.95]


*Примечание для продвинутых:* в современном numpy рекомендуется использовать вместо нее np.random.default_rng(), но seed остается стандартом во многих legacy-проектах.

####Диапазоны и последовательности

Иногда полезно генерировать массивы, состоящие из самих числовых диапазонов. Можно это делать через range, но обычно используется своя функция NumPy, которая по синтаксису точно повторяет range:

In [ ]:
np.arange(1, 10, 2) # здесь 1 - старт, 10 - стоп, 2 - шаг

array([1, 3, 5, 7, 9])

Для более упорядоченной генерации чисел с плавающей точкой есть также функция linspace, генерирующая последовательность. Ее параметрами являются старт и стоп, как у arrange, а также длина (количество чисел).

*Важное отличие*: По умолчанию конечная точка (стоп) в np.linspace включается в результат.

На выходе linspace дает числа, которые "нарезают" отрезок на равномерные части:

In [ ]:
np.linspace(1, 10, 5)

array([ 1.  ,  3.25,  5.5 ,  7.75, 10.  ])

np.linspace незаменим в Data Science и визуализации (библиотека Matplotlib), когда нужно построить гладкий график функции, например, синуса, на строго определенном отрезке: np.linspace(-np.pi, np.pi, 100). Нам не важен шаг, нам важно получить ровно 100 точек от начала до конца отрезка, чтобы график получился не угловатым.

####Округление

Если вам хочется получить числа с плавающей точкой, но не хочется возиться с огромными дробными частями, можно использовать встроенную в numpy функцию округления.

In [ ]:
raw_data = np.random.rand(2, 2)
print(raw_data)
data1 = np.round(raw_data, decimals=2)
print(data1)

[[0.90951951 0.56785466]
 [0.96541787 0.57626069]]
[[0.91 0.57]
 [0.97 0.58]]


Альтернативный вариант - сделать это через метод самого массива:

In [ ]:
print(raw_data.round(2))

[[0.91 0.57]
 [0.97 0.58]]


Если округление нужно только для красивого отображения на экране (без изменения самих данных в памяти), лучше использовать глобальную настройку отображения NumPy:

In [ ]:
np.set_printoptions(precision=2, suppress=True)

Теперь любой сгенерированный массив будет автоматически печататься с двумя знаками после точки, но при этом в памяти числа останутся точными.

In [ ]:
print(np.random.rand(3, 3))

[[0.36 0.25 0.45]
 [0.58 0.26 0.93]
 [0.7  0.44 0.31]]


####Механизм Broadcasting (Транслирование)

**Broadcasting** — это автоматическое расширение размеров массивов при выполнении арифметических операций, позволяющее складывать или умножать массивы разных форм без явного копирования данных в памяти.

Посмотрим, что будет, если мы попробуем сложить два массива разных размерностей "в лоб".

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3, 4])
arr = a + b
print(arr)

ValueError: operands could not be broadcast together with shapes (3,) (4,) 

Это и есть тот самый broadcasting: чтобы произвести операции между неравными по размеру массивами, numpy пытается превратить их в равные, но иногда у него не получается.

In [ ]:
a = np.array([1, 2, 3])
c = np.array([[1], [2], [3], [4]])
arr = a + c
print(arr) # вот так - сработает

[[2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]]


Чтобы все получилось, нужно соблюдать *Правило совместимости размеров*

Два массива совместимы для транслирования, если, начиная с самых правых размеров (с конца), их размерности:

1. Полностью совпадают.

2. Либо одна из размерностей равна 1.

Если один из массивов имеет меньше размерностей, чем другой, его форма «достраивается» слева.

Схематичный пример:

• Массив А: (3, 4)

• Массив В: (4,) => автоматически превращается в (1, 4)

• Результат: Размерности 4 и 4 совпадают, 3 и 1 совместимы (1 расширяется до 3). Операция разрешена.


In [ ]:
print(arr + b)

[[ 3  4  5]
 [ 5  6  7]
 [ 7  8  9]
 [ 9 10 11]]


####Манипуляции с формой и границами массивов

Существует несколько специальных функций numpy, позволяющих удобно производить различные манипуляции с формой и границами массивов.

np.pad обрамляет края массива новыми значениями (нулями, константами или зеркальными копиями). Это называется **паддингом**.

Зачем это нужно: Критически важно в компьютерном зрении (Computer Vision). При свертке изображений (Convolutional layers в нейросетях) картинку «обводят» рамкой из нулей, чтобы после применения фильтра размер изображения не уменьшался, а пиксели на краях тоже участвовали в расчетах.

In [ ]:
img = np.array([[1, 2],
                [3, 4]])

# Добавим рамку: по 1 строке сверху/снизу и по 1 столбцу слева/справа
padded_img = np.pad(img, pad_width=1, mode='constant', constant_values=0)
print("Массив с паддингом:\n", padded_img)

Массив с паддингом:
 [[0 0 0 0]
 [0 1 2 0]
 [0 3 4 0]
 [0 0 0 0]]


np.repeat и np.tile обе размножают данные, но делают это принципиально по-разному.

- np.repeat(arr, repeats) — копирует каждый элемент последовательно на месте.

- np.tile(arr, reps) — воспринимает весь массив как «кафельную плитку» и мостит её целиком (копирует весь массив целиком).


In [ ]:
arr = np.array([1, 2])

# Поэлементное повторение
print("np.repeat:", np.repeat(arr, 3))

# Повторение массива как плитки
print("np.tile:", np.tile(arr, 3))

np.repeat: [1 1 1 2 2 2]
np.tile: [1 2 1 2 1 2]


####Единичная матрица и эмбеддинги

Один из самых простых и самых полезных инструментов numpy - функция, генерирующая единичную матрицу - двумерный массив, заполненный единицами на указанной диагонали и нулями во всех остальных ячейках:

np.eye(n, M=None, k=0), где

• N: количество строк в матрице.

• M (опционально): количество столбцов (по умолчанию равно N, то есть получается квадратная матрица).

• k (опционально): индекс диагонали (по умолчанию 0 — главная диагональ; положительное число — верхние диагонали; отрицательное — нижние).



In [ ]:
print(np.eye(3))

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


Такая матрица называется единичной, потому что она представляет собой нейтральный элемент в умножении, то есть любая подходящая по размерам матрица, будучи умножена на единичную, дает сама себя.


В лингвистике единичные матрицы используются для базового задания **эмбеддингов**.

Тут нам понадобится немного теории.

Если у вас есть словарь размером N (например, N различных слов), каждое слово можно представить в виде векторов, которые являются столбцами или строками единичной матрицы размера N × N.

Например, для словаря из 3 слов:

	• Слово 1: [1, 0, 0]

	• Слово 2: [0, 1, 0]

	• Слово 3: [0, 0, 1]

Это базовое, «тупое» представление, где все слова ортогональны друг другу и между ними нет смысловой связи. По-простому говоря, на этом этапе (в единичной матрице) компьютер ещё ничего не знает про смысл слов. Для него каждое слово — это просто уникальный семантический признак самого себя.

Слово «кошка» — это на 100% «кошка» и на 0% всё остальное. В матрице это выглядит как [1, 0, 0].

Слово «яблоко» — это на 100% «яблоко» и на 0% «кошка». Это [0, 1, 0].

И т.д.

Матрица эмбеддингов - это плотная матрица, в которой строчки — это смысловые (семантические) векторы объектов (слов, токенов, пользователей), заданные вручную либо полученные в ходе обучения нейросети. Близкие по смыслу объекты имеют в этой матрице близкие векторы.

Допустим, мы описываем слова тремя признаками (координатами): [насколько это живое (0..1), насколько это круглое (0..1), степень абстрактности (0..1)]. В результате у нас есть три NumPy-массива (вектора) типа float для слов, допустим:

	• кошка = [1.0, 0.05, 0.0] (живое/животное, немного круглое, не абстрактное)

	• яблоко = [0.9, 1.0, 0.0] (живое/растение, круглое, не абстрактное)

	• радость = [0.0, 0.0, 1.0] (неживое, не круглое, чисто абстрактное)

Мы объединяем их в одну матрицу «словаря» размера (3, 3).

Строки единичной матрицы служат тогда исходными one-hot векторами, которые представляют собой уникальные идентификаторы токенов или слов в словаре. Умножение такого one-hot вектора на обученную матрицу весов математически работает как операция индексации (lookup), которая извлекает нужный плотный эмбеддинг из таблицы.

То есть, чтобы извлечь смысл конкретного слова, мы берем строку из единичной матрицы, соответствующую этому слову, и умножаем на матрицу весов.

In [82]:
W = np.array([[1.0, 0.05, 0.0], [0.9, 1.0, 0.0], [0.0, 0.0, 1.0]])

one_hot_cat = np.array([1, 0, 0])

# Извлекаем значение слова "кошка" математически, через умножение матриц
meaning_via_math = np.dot(one_hot_cat, W)
print(meaning_via_math)

# Сравниваем со строкой матрицы весов, извлеченной по индексу
meaning_via_index = W[0]
print(meaning_via_index)

[1.   0.05 0.  ]
[1.   0.05 0.  ]


Всеми этими вещами мы будем заниматься со вменемем, а сейчас просто примите их к сведению.

####Утилиты для работы с данными

np.unique(arr, return_counts=False) возвращает отсортированные уникальные элементы массива. Если передать return_counts=True, функция также вернет количество упоминаний каждого элемента.

In [ ]:
data = np.array([4, 1, 2, 1, 4, 4, 3])
items, counts = np.unique(data, return_counts=True)
print("Уникальные:", items)  # [1, 2, 3, 4]
print("Количество:", counts)

Уникальные: [1 2 3 4]
Количество: [2 1 1 3]


np.sort(arr) возвращает отсортированную копию массива по возрастанию.

np.argsort(arr) возвращает индексы, которые расставили бы элементы массива в отсортированном порядке. Это мощный инструмент для косвенной сортировки зависимых структур данных.

In [ ]:
weights = np.array([80, 55, 95, 70])
indices = np.argsort(weights)
print("Индексы сортировки:", indices)

Индексы сортировки: [1 3 0 2]


np.clip(arr, a_min, a_max) - идеальная утилита для обработки выбросов данных и обрезки аномалий. Вместо удаления строк она жестко «схлопывает» значения в заданные рамки [а_min, а_max].

In [ ]:
# Представим датчик, у которого рабочие границы от 0 до 100
sensor_data = np.array([-15, 45, 89, 120, 50])

# Всё, что меньше 0, станет 0. Всё, что больше 100, станет 100.
clean_data = np.clip(sensor_data, 0, 100)
print("Данные без выбросов:\n", clean_data)

Данные без выбросов:
 [  0  45  89 100  50]


На практике в Data Science значения для границ в clip часто рассчитываются динамически на основе математической статистики.

Есть два классических подхода к тому, как автоматически определить значения a_min и a_max для фильтрации аномалий:


1. Правило трех сигм (Standard Deviation)

Если наши данные распределены нормально, то 99.7% всех значений лежат в пределах трех стандартных отклонений ("сигм") от среднего арифметического. Всё, что выходит за эти рамки, считается редкой аномалией (выбросом).

Этот метод лучше всего работает, если в данных нет экстремальных единичных перекосов, которые могут сильно сместить само среднее значение.

In [ ]:
# Генерируем данные для теста
data = np.random.randn(100)
# Искусственно создаем выброс
outline = np.array([4])
data = np.concatenate((data, outline))

# Рассчитываем статистику распределения данных
mean = np.mean(data)
std = np.std(data)

# Динамически определяем границы "нормальности"
lower_bound = mean - 3 * std
upper_bound = mean + 3 * std

# "Причесываем" выбросы под статистическую норму
clean_data = np.clip(data, lower_bound, upper_bound)
print("Нормализованные данные:\n", clean_data)

Нормализованные данные:
 [ 0.29 -2.09 -1.86 -0.3   0.45  0.74 -1.59  0.64 -0.25 -0.88 -0.65  0.8
  0.27  0.05  0.24  0.93 -0.1   0.05  1.14 -0.33  0.62 -0.5  -0.42 -0.1
  0.52  1.19  0.1  -1.5   0.63  2.45  0.11 -1.71  0.17 -0.   -0.09  0.95
  1.38  0.68 -0.13  1.13 -1.49 -0.72 -0.6  -0.03  0.51 -0.2  -0.91  0.97
 -0.24  0.1   0.55  0.04  0.04  1.03  0.15  0.86  0.54  2.02  0.41  0.64
  1.1   0.06 -0.64 -0.47 -0.46 -0.95  0.32 -0.63  0.19 -0.16 -0.04  0.56
 -0.54  0.64  1.57  0.34 -0.78 -0.62 -0.25 -0.59 -0.99  0.15 -0.22  0.19
  1.19 -0.24 -1.19 -0.4   0.77  0.68  0.52 -0.62  0.75  0.98 -0.7  -0.04
  2.02 -0.74 -1.71  1.38  2.9 ]



2. Метод квантилей (Процентили)

Если данные имеют несимметричное распределение или содержат очень сильные шумы, среднее значение и сигма могут исказиться. В таких случаях используют квантили (обычно через функцию np.percentile). Например, мы можем жестко отсечь по 1% самых минимальных и самых максимальных значений (оставив центральные 98%).

Такой подход гарантирует, что даже если в данных был случайный скачок в миллион раз, он не испортит границы фильтрации, а просто аккуратно сожмется до уровня 99-го процентиля.

In [ ]:
# Находим 1-й и 99-й процентили данных
q_low = np.percentile(data, 1)   # значение, ниже которого только 1% данных
q_high = np.percentile(data, 99) # значение, выше которого только 1% данных

# Ограничиваем массив по рассчитанным квантилям
clean_data = np.clip(data, q_low, q_high)

print("Нормализованные данные:\n", clean_data)

Нормализованные данные:
 [ 0.29 -1.86 -1.86 -0.3   0.45  0.74 -1.59  0.64 -0.25 -0.88 -0.65  0.8
  0.27  0.05  0.24  0.93 -0.1   0.05  1.14 -0.33  0.62 -0.5  -0.42 -0.1
  0.52  1.19  0.1  -1.5   0.63  2.45  0.11 -1.71  0.17 -0.   -0.09  0.95
  1.38  0.68 -0.13  1.13 -1.49 -0.72 -0.6  -0.03  0.51 -0.2  -0.91  0.97
 -0.24  0.1   0.55  0.04  0.04  1.03  0.15  0.86  0.54  2.02  0.41  0.64
  1.1   0.06 -0.64 -0.47 -0.46 -0.95  0.32 -0.63  0.19 -0.16 -0.04  0.56
 -0.54  0.64  1.57  0.34 -0.78 -0.62 -0.25 -0.59 -0.99  0.15 -0.22  0.19
  1.19 -0.24 -1.19 -0.4   0.77  0.68  0.52 -0.62  0.75  0.98 -0.7  -0.04
  2.02 -0.74 -1.71  1.38  2.45]


###Сериализация и сохранение данных

####NumPy (.npy) против pickle

Когда данные нужно сохранить на диск, встает выбор между встроенным в Python модулем pickle и .npy - родным для NumPy форматом.

Сравнение:

1. Скорость: c .npy работать значительно быстрее для больших числовых массивов. NumPy сбрасывает сырой буфер памяти напрямую на диск, минуя сложную сериализацию объектов.

2. Размер: Формат .npy хранит только заголовок с метаданными (форма, тип данных) и сам непрерывный блок байт. pickle добавляет избыточную служебную информацию о Python-структурах, из-за чего файлы весят больше.

3. Безопасность: Режим pickle.load уязвим для выполнения произвольного вредоносного кода при загрузке ненадежных файлов. Формат .npy более изолирован и безопасен.

####Использование NumPy-формата:

- np.save('filename.npy', arr) — сохраняет массив в бинарном формате .npy.
- np.load('filename.npy') — быстро загружает массив обратно в память.

####Сохранение нескольких массивов в один файл (np.savez):

Очень часто в Data Science нужно сохранить не один изолированный массив, а целый пакет связанных данных. Например: обучающую выборку X_train, ответы к ней y_train, а также тестовые данные X_test и y_test.

Создавать под каждый массив отдельный .npy файл неудобно. Для этого существует функция np.savez, которая упаковывает несколько массивов в один архив с расширением .npz. Массивы внутри архива сохраняются в виде словаря («ключ — значение»).

Вот пример ее использования:






In [ ]:
# Генерируем тренировочные и тестовые датасеты
X_train = np.random.randn(1000, 10)
y_train = np.random.randint(0, 2, size=1000)
X_test  = np.random.randn(200, 10)

# Сохраняем всё в один .npz архив, задавая массивам имена-ключи
np.savez('dataset.npz', train_features=X_train, train_labels=y_train, test_features=X_test)

При загрузке такого файла мы получаем ленивый объект-контейнер. Данные не загружаются в оперативную память все сразу — NumPy прочитает конкретный массив с диска только тогда, когда мы к нему обратимся по ключу:

In [ ]:
# Загружаем архив
archive = np.load('dataset.npz')

# Смотрим, какие массивы лежат внутри архива
print("Доступные ключи:", archive.files)  # ['train_features', 'train_labels', 'test_features']

# Извлекаем нужный массив по его имени
current_train_x = archive['train_features']
print(current_train_x)

Доступные ключи: ['train_features', 'train_labels', 'test_features']
[[ 1.14 -0.46 -0.11 ...  0.88  0.46  0.58]
 [-0.48 -0.35 -1.85 ...  0.64  0.4   0.69]
 [-1.16  2.56 -1.22 ...  0.02 -0.65  0.94]
 ...
 [ 0.03 -0.08  0.67 ... -1.53  0.45 -0.23]
 [-0.7   0.77 -0.15 ... -0.54  0.46  0.79]
 [-0.39  0.2   0.19 ...  0.11 -1.36  2.14]]


Если ваши данные огромны, а на диске мало места, NumPy предлагает модификацию np.savez_compressed. Она работает абсолютно так же, как и savez, но дополнительно применяет алгоритм сжатия к файлам внутри архива.

In [ ]:
np.savez_compressed('dataset_compressed.npz', train_features=X_train, train_labels=y_train)

*Плюс:* Размер итогового файла на диске может уменьшиться в несколько раз (особенно если в массивах много нулей или повторяющихся значений).

*Минус:* Запись и чтение будут занимать чуть больше времени, так как процессору требуется тратить ресурсы на архивацию и разархивацию «на лету».